# Hesitation: think time and hand hovering as teaching signals

**Deliverable:** input features for the player model (README Teaching: *"handling pressure"*,
*"decision making"*). A physical board gives signals an online platform cannot: how long the
human thinks, and where their hand hovers before committing (a hand that hovers over the queen and
then moves a pawn says something about the decision).

Two sources:

* **Think time** per move: from the moment the robot's move is verified to the moment the human's
  move is seen as stable (`MoveEvent.think_time_s` from `game_manager.ipynb`). Implemented.
* **Hover events** from vision: `gambit_vision` reports when a hand is over the board and,
  ideally, over which square. Data structures defined here; extraction is TODO.

**TODO checklist**
- [x] Data structures, per-move think-time features, listener that collects them.
- [ ] Vision side: hand detection + hand-to-square mapping -> `HoverEvent`s (agree format with gambit_vision).
- [ ] `hesitation_score`: think time relative to the player's own baseline and to position
      complexity (number of legal moves, eval spread of the top multipv lines).
- [ ] Decide which features the player model uses (habits.ipynb) and validate that they predict
      mistakes at all before relying on them.

In [ ]:
from __future__ import annotations

import statistics
from dataclasses import dataclass, field
from typing import Optional

import chess

## Data structures

In [ ]:
@dataclass
class HoverEvent:
    t_start: float
    t_end: float
    square: Optional[chess.Square] = None     # None: hand over the board but square unknown
    touched: bool = False                     # piece lifted/touched (if vision can tell)

    @property
    def duration(self) -> float:
        return max(0.0, self.t_end - self.t_start)


@dataclass
class MoveTiming:
    ply: int
    by: str
    think_time_s: float
    hovers: list = field(default_factory=list)       # HoverEvents during this think
    n_legal_moves: int = 0

    @property
    def hovered_squares(self) -> list:
        return [h.square for h in self.hovers if h.square is not None]

    @property
    def hover_time_s(self) -> float:
        return sum(h.duration for h in self.hovers)


@dataclass
class HesitationFeatures:
    ply: int
    think_time_s: float
    think_ratio_to_median: float      # think time / median human think time so far (1.0 = typical)
    n_hovers: int
    hover_time_s: float
    distinct_hovered_squares: int
    n_legal_moves: int

## Collector

`HesitationTracker` is a game-manager listener (`listeners=[tracker]`). Hover events from vision
are pushed with `add_hover` while the human thinks; they are attached to the next human move.

In [ ]:
class HesitationTracker:
    def __init__(self):
        self.timings: list[MoveTiming] = []
        self._pending_hovers: list[HoverEvent] = []

    def add_hover(self, ev: HoverEvent) -> None:
        self._pending_hovers.append(ev)

    def __call__(self, event) -> None:          # event: game_manager.MoveEvent
        board = chess.Board(event.fen_before)
        t = MoveTiming(event.ply, event.by, event.think_time_s, n_legal_moves=board.legal_moves.count())
        if event.by == "human":
            t.hovers, self._pending_hovers = self._pending_hovers, []
        self.timings.append(t)

    def human_timings(self) -> list[MoveTiming]:
        return [t for t in self.timings if t.by == "human"]

    def features(self) -> list[HesitationFeatures]:
        out, seen = [], []
        for t in self.human_timings():
            seen.append(t.think_time_s)
            med = statistics.median(seen)
            out.append(HesitationFeatures(
                t.ply, t.think_time_s, t.think_time_s / med if med > 0 else 1.0,
                len(t.hovers), t.hover_time_s, len(set(t.hovered_squares)), t.n_legal_moves))
        return out

## TODO: hover extraction and a hesitation score

In [ ]:
def hovers_from_observations(observations: list) -> list[HoverEvent]:
    """Turn a stream of vision observations (BoardObservation with hand_present and, later, a hand
    square) into HoverEvents.

    TODO: needs a hand position from gambit_vision (not in BoardObservation yet). Merge consecutive
    frames over the same square into one event; drop events shorter than ~0.2 s (noise).
    """
    raise NotImplementedError("hovers_from_observations: needs hand-square output from vision")


def hesitation_score(features: HesitationFeatures, baseline: dict) -> float:
    """How unusually long/indecisive this move was for this player (0 = typical).

    TODO: e.g. z-score of log(think time) against the player's history (`baseline` = mean/std of
    log think time per phase), plus a term for hovering over several pieces. Calibrate on logs.
    """
    raise NotImplementedError("hesitation_score")

## Self-check

In [ ]:
from types import SimpleNamespace

tr = HesitationTracker()
tr.add_hover(HoverEvent(0.0, 1.5, chess.D1))
tr.add_hover(HoverEvent(2.0, 2.5, chess.E2))
tr(SimpleNamespace(ply=0, by="human", fen_before=chess.STARTING_FEN, think_time_s=10.0))
tr(SimpleNamespace(ply=1, by="robot", fen_before=chess.STARTING_FEN, think_time_s=0.5))
tr(SimpleNamespace(ply=2, by="human", fen_before=chess.STARTING_FEN, think_time_s=30.0))
f = tr.features()
assert len(f) == 2 and f[0].n_hovers == 2 and abs(f[0].hover_time_s - 2.0) < 1e-9
assert f[0].distinct_hovered_squares == 2 and f[0].n_legal_moves == 20
assert f[1].n_hovers == 0 and abs(f[1].think_ratio_to_median - 30 / 20) < 1e-9
print("hesitation self-check passed")